# General functioning of flexibility provision

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import pypsa

# Add the scripts/pypsa-de directory to the path
scripts_path = os.path.join(os.path.dirname(os.getcwd()), "scripts", "pypsa-de")
sys.path.append(scripts_path)

# plotting
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import seaborn as sns
from flexibility_analysis import (
    calc_flexibility_contributions,
    calc_flexibility_needs,
    calc_residual_load,
    calc_supply_demand,
    expand_to_1h,
)
from flexibility_utils import tech_colors, year_colors_gradient
from matplotlib.colors import rgb2hex
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

font1 = {"fontname": "Arial"}

# overwrite year colors
year_colors_gradient = {
    year: rgb2hex(plt.cm.viridis(np.linspace(0, 1, 7))[i])
    for i, year in enumerate([2020, 2025, 2030, 2035, 2040, 2045, 2050])
}

kwargs = {
    "groupby": ["bus", "carrier"],
    "at_port": True,
    "nice_names": False,
}

In [ ]:
f = "/home/julian-geis/Documents/06_PhD/02Flexibility/runs/"
run = "20251104-flex-4scenarios-27cl-1H" #"20251015-flex-scenario-update-27cl-3h"  # "20251013_flex-scenarios-temporal-industry-load-27cl-3h"

scenarios = ["LowBatt50", "MedFlex", "HigFlex"]
years = [2025, 2035, 2045]

PLOT_DIR = f + run + "/plots/flex_functioning/"
for s in scenarios:
    os.makedirs(PLOT_DIR + s + "/", exist_ok=True)

In [ ]:
networks = {}
for scenario in scenarios:
    for year in years:
        fn = f"{f}/{run}/{scenario}/networks/base_s_27__none_{year}.nc"
        networks[(scenario, year)] = pypsa.Network(fn)

# Functions

In [ ]:
def plot_generation_profiles(
    carriers,
    electricity_supply,
    electricity_demand,
    font1=None,
    figsize=(16, 9),
    save_path=None,
):
    """
    Plot generation profiles of non-dispatchable supply and demand as heatmaps with monthly generation lines.

    Parameters
    ----------
    carriers : list of lists
        List of carrier groups to plot, e.g., [["onwind"], ["solar", "solar rooftop"]]
    electricity_supply : pd.DataFrame
        DataFrame with electricity supply data
    electricity_demand : pd.DataFrame
        DataFrame with electricity demand data
    font1 : dict, optional
        Font properties dictionary for labels
    figsize : tuple, optional
        Figure size as (width, height)
    """

    if font1 is None:
        font1 = {}

    fig, axs = plt.subplots(nrows=int(len(carriers) / 2), ncols=2, figsize=figsize)

    for i, ax in enumerate(axs.reshape(-1)):
        if carriers[i][0] in electricity_supply.index:
            df = electricity_supply.loc[carriers[i]].sum() / 1e3  # in GW
        elif carriers[i][0] in electricity_demand.index:
            df = electricity_demand.loc[carriers[i]].sum() / 1e3  # in GW
        else:
            print(f"{carriers[i][0]} does not exist!")
            continue

        hours = df.index.hour.unique()[::-1]
        df_start = pd.DataFrame(index=pd.Index(df.index.date).unique())
        for hour in hours:
            df_start[str(hour)] = df[df.index.hour == hour].values

        sns.heatmap(
            df_start.transpose(),
            ax=ax,
            cmap=plt.get_cmap("magma_r"),
            linewidth=0.001,
            xticklabels=False,
            cbar_kws={"label": "Power in GW", "pad": 0.1},
        )

        ax.set_title(f"{carriers[i]}", fontsize=18, **font1)
        ax.set_ylabel("hour of the day", fontsize=12, **font1)
        ax.set_xlabel("day of the year", fontsize=12, **font1)

        # Set x-ticks to show only month starts
        month_starts = pd.date_range(start="2019-01-01", end="2019-12-31", freq="MS")
        tick_positions = [(d - pd.Timestamp("2019-01-01")).days for d in month_starts]
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([d.strftime("%b") for d in month_starts])

        # Plot monthly generation as horizontal lines
        df_monthly = df.resample("ME").sum() / 1e3  # in TWh

        ax2 = ax.twinx()
        for j, (date, value) in enumerate(df_monthly.items()):
            # Get start and end day of month
            month_start = (date.replace(day=1) - pd.Timestamp("2019-01-01")).days
            month_end = (
                (date + pd.DateOffset(days=1)).replace(day=1)
                - pd.Timestamp("2019-01-01")
            ).days - 1

            ax2.hlines(
                value,
                month_start,
                month_end,
                color="white",
                lw=2,
                path_effects=[pe.Stroke(linewidth=3, foreground="black"), pe.Normal()],
            )

        # Create custom legend handle with path effects
        legend_line = Line2D(
            [0],
            [0],
            color="white",
            lw=2,
            path_effects=[pe.Stroke(linewidth=3, foreground="black"), pe.Normal()],
        )
        ax2.legend(
            handles=[legend_line], labels=["Generation (right axis)"], loc="lower right"
        )
        ax2.set_ylabel("Generation in TWh (monthly sum)", fontsize=12, **font1)
        ax2.grid(False)

    fig.tight_layout(pad=3)
    if save_path:
        fig.savefig(save_path, dpi=300)
    plt.show()

    return fig, axs


def plot_loads_by_carrier(
    n, tech_colors, country_code="DE", figsize=(12, None), save_path=None
):
    """
    Plot load profiles clustered by energy carrier type.

    Parameters
    ----------
    n : pypsa.Network
        PyPSA network object
    tech_colors : dict
        Dictionary mapping carrier names to colors
    country_code : str
        Country code to filter (e.g., 'DE')
    figsize : tuple
        Figure size (width, height). If height is None, auto-calculated.
    """
    import matplotlib.pyplot as plt

    # Define carrier groupings based on keywords
    carrier_groups = {
        "Electricity": ["electricity", "electric", "EV"],
        "Heat": ["heat"],
        "Other": [
            "oil",
            "H2",
            "fuel cell",
            "gas",
            "coal",
            "kerosene",
            "methanol",
            "naphtha",
            "biomass",
        ],
    }

    # Cluster carriers into groups
    clustered = {group: [] for group in carrier_groups}
    for carrier in n.loads["carrier"].unique():
        for group, keywords in carrier_groups.items():
            if any(kw in carrier.lower() for kw in keywords):
                clustered[group].append(carrier)
                break

    # Remove empty groups
    clustered = {k: v for k, v in clustered.items() if v}

    # Create subplots
    n_plots = len(clustered)
    if figsize[1] is None:
        figsize = (figsize[0], n_plots * 3)

    fig, axs = plt.subplots(nrows=n_plots, ncols=1, figsize=figsize)
    if n_plots == 1:
        axs = [axs]

    for ax, (group, carriers) in zip(axs, clustered.items()):
        for carrier in carriers:
            ind = n.loads[
                (n.loads["carrier"] == carrier)
                & (n.loads.bus.str.startswith(country_code))
            ].index
            if len(ind) > 0:
                load_profile = n.loads_t.p[ind].sum(axis=1) / 1e3  # in GW
                color = tech_colors.get(carrier, None)
                ax.plot(load_profile, label=carrier, alpha=0.8, color=color)

        ax.set_title(f"{group} Loads", fontsize=14, fontweight="bold")
        ax.set_ylabel("Load (GW)")
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=9)
        ax.grid(True, alpha=0.3)

    axs[-1].set_xlabel("Time")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300)
    plt.show()

    return fig, axs


def plot_residual_load_months(
    networks,
    scenario,
    years,
    month1=2,
    month2=6,
    country_code="DE",
    figsize=(16, 10),
    save_path=None,
):
    """
    Plot residual load and renewable penetration for two selected months across multiple years.

    Parameters
    ----------
    networks : dict
        Dictionary of PyPSA networks with keys (scenario, year)
    scenario : str
        Scenario name
    years : list
        List of years to plot
    month1, month2 : int
        Month numbers (1-12) to plot
    country_code : str
        Country code to filter
    figsize : tuple
        Figure size
    """
    import calendar

    import matplotlib.pyplot as plt
    import pandas as pd

    # Storage for results
    residual_loads = {}
    demands = {}
    renewable_gens = {}

    # Calculate residual load for each year
    for year in years:
        print(f"Processing year {year}...")

        # Get supply and demand data
        s, d = calc_supply_demand(
            networks[(scenario, year)],
            energy=False,
            interconnectors=False,
            merge_dist_grid=True,
            drop_dist_grid=True,
            add_diff_as_import=True,
        )

        # Expand to 1H
        electricity_supply = expand_to_1h(s, unit="power")
        electricity_demand = expand_to_1h(d, unit="power")

        # Calculate residual load
        residual_load = calc_residual_load(electricity_supply, electricity_demand)

        # Store results (convert to GW)
        residual_loads[year] = residual_load / 1e3  # Convert MW to GW
        demands[year] = electricity_demand.loc[
            ["electricity", "agriculture electricity", "industry electricity"]
        ].sum()
        renewable_gens[year] = electricity_supply.loc[
            [
                "onwind",
                "offwind-ac",
                "offwind-dc",
                "solar",
                "solar-hsat",
                "solar rooftop",
                "ror",
            ]
        ].sum()

    # Create subplots
    fig, axs = plt.subplots(2, 2, figsize=figsize)
    months = [month1, month2]
    month_names = [calendar.month_name[m] for m in months]

    # Collect data for y-axis limits
    res_load_data = []
    penetration_data = []

    for col, (month, month_name) in enumerate(zip(months, month_names)):
        month_str = f"2019-{month:02d}"

        # Plot residual load for this month
        for i, year in enumerate(years):
            if month_str in residual_loads[year].index:
                data = residual_loads[year][month_str]
                res_load_data.extend(data.values)
                axs[0, col].plot(
                    data.index,
                    data.values,
                    color=year_colors_gradient[year],
                    linewidth=2,
                    label=str(year),
                    alpha=0.8,
                )

        axs[0, col].set_title(
            f"Residual Load - {month_name}", fontsize=14, fontweight="bold"
        )
        axs[0, col].set_ylabel("Residual Load (GW)", fontsize=12)
        axs[0, col].grid(True, alpha=0.3)
        axs[0, col].legend(title="Year")

        # Set x-ticks to week starts only
        month_start = pd.Timestamp(f"2019-{month:02d}-01")
        month_end = (month_start + pd.DateOffset(months=1)) - pd.Timedelta(days=1)
        week_starts = pd.date_range(start=month_start, end=month_end, freq="W-MON")
        # Include first day of month if it's not a Monday
        if month_start.dayofweek != 0:
            week_starts = pd.DatetimeIndex([month_start]).append(week_starts)
        axs[0, col].set_xticks(week_starts)
        axs[0, col].set_xticklabels([d.strftime("%d %b") for d in week_starts])

        # Plot renewable penetration for this month
        for i, year in enumerate(years):
            if (
                month_str in demands[year].index
                and month_str in renewable_gens[year].index
            ):
                demand_data = demands[year][month_str]
                renewable_data = renewable_gens[year][month_str]
                penetration = renewable_data / demand_data * 100
                penetration_data.extend(penetration.values)
                axs[1, col].plot(
                    penetration.index,
                    penetration.values,
                    color=year_colors_gradient[year],
                    linewidth=2,
                    label=str(year),
                    alpha=0.8,
                )

        axs[1, col].set_title(
            f"Renewable Penetration - {month_name}", fontsize=14, fontweight="bold"
        )
        axs[1, col].set_ylabel("Renewables / Loads (%)", fontsize=12)
        axs[1, col].set_xlabel("Time", fontsize=12)
        axs[1, col].grid(True, alpha=0.3)
        axs[1, col].legend(title="Year")

        # Set x-ticks to week starts only
        axs[1, col].set_xticks(week_starts)
        axs[1, col].set_xticklabels([d.strftime("%d %b") for d in week_starts])

    # Set equal y-axis limits for each row
    if res_load_data:
        y_min, y_max = min(res_load_data), max(res_load_data)
        y_margin = (y_max - y_min) * 0.05
        for col in range(2):
            axs[0, col].set_ylim(y_min - y_margin, y_max + y_margin)

    if penetration_data:
        y_min, y_max = min(penetration_data), max(penetration_data)
        y_margin = (y_max - y_min) * 0.05
        for col in range(2):
            axs[1, col].set_ylim(y_min - y_margin, y_max + y_margin)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300)
    plt.show()

    return fig, axs


# Residual load duration curve:
def plot_residual_load_duration_curve(
    networks, scenario, years, country_code="DE", figsize=(8, 8), save_path=None
):
    """
    Plot residual load duration curve for multiple years.

    Parameters
    ----------
    networks : dict
        Dictionary of PyPSA networks with keys (scenario, year)
    scenario : str
        Scenario name
    years : list
        List of years to plot
    country_code : str
        Country code to filter
    figsize : tuple
        Figure size
    """
    import matplotlib.pyplot as plt
    import numpy as np

    fig, ax = plt.subplots(figsize=figsize)

    for year in years:
        print(f"Processing year {year}...")

        # Get supply and demand data
        s, d = calc_supply_demand(
            networks[(scenario, year)],
            energy=False,
            interconnectors=False,
            merge_dist_grid=True,
            drop_dist_grid=True,
            add_diff_as_import=True,
        )

        # Expand to 1H
        electricity_supply = expand_to_1h(s, unit="power")
        electricity_demand = expand_to_1h(d, unit="power")

        # Calculate residual load
        residual_load = calc_residual_load(electricity_supply, electricity_demand)

        # Convert to GW and sort in descending order
        residual_load_gw = residual_load / 1e3
        sorted_residual = np.sort(residual_load_gw.values)[::-1]

        # Create hours array
        hours = np.arange(len(sorted_residual))

        # Plot
        ax.plot(
            hours,
            sorted_residual,
            color=year_colors_gradient[year],
            linewidth=1.5,
            label=str(year),
            alpha=0.9,
        )

    ax.axhline(0, color="black", linestyle="-", linewidth=0.8, alpha=0.5)
    ax.set_title("Sorted Residual Load Duration Curve", fontsize=14, fontweight="bold")
    ax.set_xlabel("Hours of a year [h]", fontsize=12)
    ax.set_ylabel("[GW]", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(title="Year", fontsize=11, title_fontsize=12)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300)
    plt.show()

    return fig, ax


def plot_flexibility_contributions(
    flex_df, tech_colors, year, figsize=(15, 6), save_path=None
):
    """
    Plot flexibility contributions by technology and granularity with dual percentage labels.
    Upper %: Share of granularity total
    Lower %: Share of technology total
    """

    # Reset index and parse technology info
    df = flex_df.reset_index()
    df["Type"] = df["Technology"].str.split("_").str[0]
    df["Tech_name"] = df["Technology"].str.split("_", n=1).str[1]
    df = df[df["Tech_name"] != "agriculture electricity"]  # Exclude

    # Pivot and reorder
    combined_df = df.pivot(
        index="Granularity", columns="Tech_name", values="Contribution (TWh/year)"
    ).fillna(0)
    granularity_order = ["daily", "weekly", "monthly", "annual"]
    combined_df = combined_df.reindex(granularity_order)

    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    colors = [tech_colors.get(tech, "gray") for tech in combined_df.columns]
    combined_df.plot(kind="barh", stacked=True, ax=ax, color=colors, width=0.7)

    # Create text outline effect for better readability
    outline_effect = [pe.withStroke(linewidth=3, foreground="black")]

    # Add dual percentage labels
    for i, granularity in enumerate(granularity_order):
        row = combined_df.loc[granularity]
        granularity_total = row.abs().sum()  # Total for this granularity
        x_neg, x_pos = 0, 0

        for tech in combined_df.columns:
            value = row[tech]

            # Only add labels if value is significant
            if abs(value) >= 5:
                # Calculate both percentages
                tech_total = combined_df[tech].abs().sum()  # Total for this technology
                tech_percentage = abs(value) / tech_total * 100  # % of technology total
                gran_percentage = (
                    abs(value) / granularity_total * 100
                )  # % of granularity total

                # Calculate position
                x_pos_label = (x_neg + value / 2) if value < 0 else (x_pos + value / 2)

                # Upper text: granularity percentage (white with black outline)
                text1 = ax.text(
                    x_pos_label,
                    i + 0.15,
                    f"{gran_percentage:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=8,
                    fontweight="bold",
                    color="white",
                )
                text1.set_path_effects(outline_effect)

                # Lower text: technology percentage (white with black outline)
                text2 = ax.text(
                    x_pos_label,
                    i - 0.15,
                    f"{tech_percentage:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=8,
                    fontweight="bold",
                    color="white",
                )
                text2.set_path_effects(outline_effect)

            # Always update position (even for bars without labels)
            if value < 0:
                x_neg += value
            else:
                x_pos += value

    # Combine all legends into one
    supply_techs = df[df["Type"] == "Supply"]["Tech_name"].unique()
    demand_techs = df[df["Type"] == "Demand"]["Tech_name"].unique()

    # Build combined legend handles
    combined_handles = []

    # Add supply technologies
    if len(supply_techs) > 0:
        combined_handles.append(
            Patch(facecolor="none", edgecolor="none", label="Supply Technologies")
        )
        for t in combined_df.columns:
            if t in supply_techs:
                combined_handles.append(
                    Patch(facecolor=tech_colors.get(t, "gray"), label=f"  {t}")
                )

    # Add demand technologies
    if len(demand_techs) > 0:
        combined_handles.append(
            Patch(facecolor="none", edgecolor="none", label="")
        )  # Spacer
        combined_handles.append(
            Patch(facecolor="none", edgecolor="none", label="Demand Technologies")
        )
        for t in combined_df.columns:
            if t in demand_techs:
                combined_handles.append(
                    Patch(facecolor=tech_colors.get(t, "gray"), label=f"  {t}")
                )

    # Add percentage labels explanation
    combined_handles.append(
        Patch(facecolor="none", edgecolor="none", label="")
    )  # Spacer
    combined_handles.append(
        Patch(facecolor="none", edgecolor="none", label="Percentage Labels")
    )
    combined_handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="white",
            markeredgecolor="black",
            markeredgewidth=1.5,
            markersize=8,
            label="  % of granularity total",
        )
    )
    combined_handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="white",
            markeredgecolor="black",
            markeredgewidth=1.5,
            markersize=8,
            label="  % of technology total",
        )
    )

    # Create single legend in upper left
    ax.legend(handles=combined_handles, loc="upper left", fontsize=8, framealpha=0.9)

    # Formatting
    ax.set_ylabel("Temporal Granularity", fontsize=11)
    ax.set_xlabel("Contribution (TWh/year)", fontsize=11)
    ax.grid(True, alpha=0.3, axis="x")
    ax.axvline(0, color="black", linewidth=0.8)

    # Extend x-axis to the right
    xlim = ax.get_xlim()
    ax.set_xlim(xlim[0], xlim[1] * 1.1)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    return fig, ax

# Flexibility drivers

In [ ]:
scenario = "MedFlex"
year = 2045

In [ ]:
# undispatchable supply and undispatchable demand
non_dispatchable_supply_carriers = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar-hsat",
    "solar rooftop",
    "ror",
]
non_dispatchable_demand_carriers = [
    "electricity",
    "agriculture electricity",
    "industry electricity",
]

s, d = calc_supply_demand(
    networks[(scenario, year)],
    energy=False,
    interconnectors=False,
    merge_dist_grid=True,
    drop_dist_grid=True,
    add_diff_as_import=True,
)

electricity_supply = expand_to_1h(s, unit="power")
electricity_demand = expand_to_1h(d, unit="power")

## Non-dispatchable supply

In [ ]:
carriers = [
    ["onwind"],
    ["solar", "solar rooftop", "solar-hsat"],
    ["offwind-ac", "offwind-dc"],
    ["ror"],
]

fig, axs = plot_generation_profiles(
    carriers,
    electricity_supply,
    electricity_demand,
    font1=font1,
    save_path=PLOT_DIR + f"non_disp_supply_heatmap_{scenario}_{year}.png",
)

## Loads

In [ ]:
n = networks[(scenario, year)]
fig, axs = plot_loads_by_carrier(
    n,
    tech_colors,
    country_code="DE",
    save_path=PLOT_DIR + scenario + "/" + f"loads_by_carrier_{scenario}_{year}.png",
)

In [ ]:
# heat_loads_ind_de = n.loads[(n.loads["carrier"].str.contains("heat")) & (n.loads.bus.str.startswith("DE"))].index
# n.loads_t.p[heat_loads_ind_de][:8*7].plot()

## Residual load and renewable penetration:

In [ ]:
years = [2025, 2035, 2045]
fig, axs = plot_residual_load_months(
    networks,
    scenario,
    years,
    month1=2,
    month2=7,
    country_code="DE",
    save_path=PLOT_DIR + f"residual_load_months_{scenario}.png",
)

In [ ]:
# Residual load duration curve:
years = [2025, 2035, 2045]
fig, ax = plot_residual_load_duration_curve(
    networks,
    scenario,
    years,
    country_code="DE",
    save_path=PLOT_DIR
    + scenario
    + "/"
    + f"residual_load_duration_curve_{scenario}.png",
)

In [ ]:
# more analysis for residual load
residual_loads = {}

for year in years:
    # Get supply and demand data
    s, d = calc_supply_demand(
        networks[(scenario, year)],
        energy=False,
        interconnectors=False,
        merge_dist_grid=True,
        drop_dist_grid=True,
        add_diff_as_import=True,
    )

    # Expand to 1H
    electricity_supply = expand_to_1h(s, unit="power")
    electricity_demand = expand_to_1h(d, unit="power")

    # Calculate residual load
    residual_loads[year] = calc_residual_load(electricity_supply, electricity_demand)

In [ ]:
for year in years:
    print(year, (residual_loads[year] / 1e3).describe()[["min", "max"]])

# 1H Base 2045: min -260; max 81

In [ ]:
# fluctuations between consecutive hours
for year in years:
    print(year, residual_loads[year].diff().abs().max() / 1e3)

# 1H Base 2035: 111 / 2045: 122

In [ ]:
# calc maximum power needed for smoothing

# (residual_loads[year][:6]/1e3).plot(figsize=(20, 6), label='Residual Load')
# (residual_loads[year][:6]/1e3).rolling(window=2).mean().plot(figsize=(20, 6), ls='--', label='Rolling Mean')

max_diffs = pd.DataFrame(index=range(1, 100), columns=years)
for year in years:
    for i in range(1, 100):
        max_diffs.loc[i, year] = (residual_loads[year] / 1e3).rolling(window=i).mean().diff().abs().max()

fig, ax = plt.subplots(figsize=(10, 6))
plt.plot(max_diffs)


In [ ]:
(residual_loads[year] / 1e3).rolling(window=10).mean().diff().abs().max()

In [ ]:
def calculate_flexibility_needs(residual_load, granularity, sub_granularity=None):
    avg_at_granularity = residual_load.resample(granularity).transform("mean")
    
    if sub_granularity is not None:
        avg_at_sub_granularity = residual_load.resample(sub_granularity).transform("mean")
        flex_ts = avg_at_sub_granularity - avg_at_granularity
    else:
        flex_ts = residual_load - avg_at_granularity
    
    flex = 0.5 * flex_ts.abs().sum()
    return flex, flex_ts

In [ ]:
res = pd.DataFrame(index=range(1, 8760), columns=years)

for i in res.index:
    s, ts = calculate_flexibility_needs(residual_loads[year], granularity=f"{i}h")
    res.loc[i, year] = ts.max()

(res[year]/1e3).plot()

In [ ]:
res = pd.DataFrame(index=range(1, 1000), columns=years)

for i in res.index:
    s, ts = calculate_flexibility_needs(residual_loads[year], granularity=f"{i+1}h", sub_granularity=f"{i}h")
    res.loc[i, year] = ts.max()

(res[year]/1e3).plot(ylabel="GW", xlabel="t in hours")

In [ ]:
# Michas idea ?
res = pd.DataFrame(index=range(1, 1000), columns=years)

for i in res.index:
    s, ts = calculate_flexibility_needs(residual_loads[year], granularity=f"{2*i}h", sub_granularity=f"{i}h")
    res.loc[i, year] = ts.max()

(res[year]/1e3).plot(ylabel="GW", xlabel="t in hours")

In [ ]:
# Claude: https://claude.ai/share/70414d20-ee93-4e8e-8f28-f5d9707e9e71
load = residual_loads[year] / 1e3
# Calculate max power for different time horizons
horizons = range(1, 101)  # 1 to 100 hours
max_powers = []

for H in horizons:
    # Rolling mean
    smoothed = np.convolve(load, np.ones(H)/H, mode='valid')
    # Deviation
    deviation = load[H-1:] - smoothed
    # Max absolute deviation
    max_power = np.abs(deviation).max()
    max_powers.append(max_power)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(horizons, max_powers, linewidth=2)
plt.xlabel('Time Horizon (hours)')
plt.ylabel('Max Power Needed (GW)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
residual_load = residual_loads[year]
granularity = "24h"
sub_granularity = None

avg_at_granularity = residual_load.resample(granularity).transform("mean")

if sub_granularity is not None:
    avg_at_sub_granularity = residual_load.resample(sub_granularity).transform("mean")
    flex_ts = avg_at_sub_granularity - avg_at_granularity
else:
    flex_ts = residual_load - avg_at_granularity


fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(residual_load[:24] / 1e3, label="Residual Load", alpha=0.5)
ax.plot(avg_at_granularity[:24] / 1e3, label=f"Average at {granularity}", linewidth=2)
ax.plot(flex_ts[:24] / 1e3, label="Flexibility Need TS", linewidth=1)
ax.set_ylabel("Power (GW)")
ax.legend()
plt.show()

In [ ]:
ts[:10].plot(marker="x")

In [ ]:
assert 0

# Flex needs analysis

In [ ]:
scenario = "MediumFlex"
year = 2045

# Extract supply and demand data
s, d = calc_supply_demand(
    networks[(scenario, year)],
    energy=False,
    interconnectors=False,
    merge_dist_grid=True,
    drop_dist_grid=True,
    add_diff_as_import=True,
)
electricity_supply = expand_to_1h(s, unit="power")
electricity_demand = expand_to_1h(d, unit="power")

residual_load = calc_residual_load(electricity_supply, electricity_demand)
calc_flexibility_needs(residual_load)

In [ ]:
# maximum daily flex needs

# Calculate flexibility causes and contributions (raw)
scenario = "MediumFlex"

flex_causes_raw = {}
flex_contributions_raw = {}

for year in [2025, 2035, 2045]:
    # Extract supply and demand data
    s, d = calc_supply_demand(
        networks[(scenario, year)],
        energy=False,
        interconnectors=False,
        merge_dist_grid=True,
        drop_dist_grid=True,
        add_diff_as_import=True,
    )
    electricity_supply = expand_to_1h(s, unit="power")
    electricity_demand = expand_to_1h(d, unit="power")

    residual_load = calc_residual_load(electricity_supply, electricity_demand)

    daily_avg = residual_load.resample("D").transform("mean")
    weekly_avg = residual_load.resample("W").transform("mean")
    monthly_avg = residual_load.resample("MS").transform("mean")
    annual_avg = residual_load.resample("YS").transform("mean")

    print(f"Year: {year}")
    daily_needs = (
        ((residual_load - daily_avg).abs() / 2).resample("D").mean() * 24 / 1e3
    )  # in GWh/day

    fig, ax = plt.subplots(figsize=(12, 5))
    daily_needs.plot(ax=ax)
    daily_needs.rolling(window=7).mean().plot(ax=ax)
    daily_needs.rolling(window=30).mean().plot(ax=ax)
    ax.set_title(f"Daily Flexibility Needs ({year})", fontsize=14, fontweight="bold")
    ax.set_ylabel("Daily Flexibility Needs (GWh/day)", fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    print(daily_needs.describe())
    print()

In [ ]:
# Maximum daily need is almost 1000 GWh/day
# to shift that you would need techs with 1000 / 24 = 41.67 GW capacity if it is evenly distributed over the day


In [ ]:
n = networks[(scenario, year)]

In [ ]:
n.buses_t.p.filter(regex=r"DE0 \d+$")

In [ ]:
kwargs = {
    "groupby": ["name", "bus", "carrier"],
    "nice_names": False,
}

n.statistics.withdrawal(
    bus_carrier=["AC", "low voltage"],
    aggregate_time=False,
    **kwargs,
).filter(like="DE", axis=0).groupby(level="bus").sum()

In [ ]:
daily_diff = residual_load - daily_avg

fig, ax = plt.subplots(figsize=(12, 5))
residual_load.plot(ax=ax)
daily_diff.plot(ax=ax)
daily_avg.plot(ax=ax)

In [ ]:
# Calculate flexibility causes and contributions (raw)
scenario = "MediumFlex"

flex_causes_raw = {}
flex_contributions_raw = {}

for year in years:
    # Extract supply and demand data
    s, d = calc_supply_demand(
        networks[(scenario, year)],
        energy=False,
        interconnectors=False,
        merge_dist_grid=True,
        drop_dist_grid=True,
        add_diff_as_import=True,
    )
    electricity_supply = expand_to_1h(s, unit="power")
    electricity_demand = expand_to_1h(d, unit="power")

    # Calculate flexibility contributions
    flexible_df, inflexible_df = calc_flexibility_contributions(
        electricity_supply=electricity_supply,
        electricity_demand=electricity_demand,
        granularity="all",
        analyze="both",
        print_info=False,
    )

    flex_causes_raw[year] = inflexible_df
    flex_contributions_raw[year] = flexible_df

In [ ]:
# flex contributions plot
year = 2045
fig, ax = plot_flexibility_contributions(
    flex_causes_raw[year],
    tech_colors,
    year,
    save_path=PLOT_DIR + scenario + "/" + f"flex_causes_detailed_{scenario}_{year}.png",
)

# Flex provision

In [ ]:
# calc maximum soaking up of electricity from PtH / PtX / BEV / all together

# Flex provision functioning

# Hourly analysis

In [ ]:
mix_1H = {}
for year in [2020, 2030, 2035, 2040, 2045, 2050]:
    mix_1H[year] = pypsa.Network(
        f"//home/julian-geis/Documents/06_PhD/02Flexibility/runs/20250806_1H_27cl/KN2045_Mix/networks/base_s_27__none_{year}.nc"
    )